# TSM Development-only Snapshot preparation — no backtest

Run cells top to bottom in a **fresh Colab runtime**. This is one bounded inventory → producer → data-only preflight, not a full-history validation. No strategy, batch reservation or signing API is called. Original Snapshot, authorization, Holdout and costs stay unchanged.

Set explicit **TSM native Parquet upstream roots** if known. Empty roots do **not** prove chunks absent. Inventory reads metadata only. If no safe input is selected, the known normalized file is planned only. Mixed-source conversion requires a personally approved exact plan; the notebook never grants that approval. Inspect `producer_plan.json`, including full-file hash bytes and compressed decoded column extents. Stop here and obtain lead permission before supplying an approval file.

After lawful production, this notebook creates actual Manifest/Receipt/lineage and performs real Core loading/15m/PIT checks. The NEW research authorization remains unsigned. No real SMA run becomes eligible until its personal approval and normal Agent checks. Outputs are exclusive/new; reruns do not overwrite artifacts. To resume with an existing derived artifact, set its binding path and relocated root if necessary.


In [ ]:
from pathlib import Path
CORE_SHA = "b16179a7cc077c36046574769a32c367fa476cae"
AGENT_SHA = "9e368f8925a425f7dae8921291c168b5d197f33a"
SOURCE_ROOT = Path("/content/drive/MyDrive/00Quant_Research/data/00_Quant_Research_Hub/20261101_205431_06359faf7ef4_IBKR_FINAL_GATE_FULL")
PARENT_MANIFEST_SHA = "d960602cff24f7350b056d69e19d8e0b8734cdad266d3782bc56ebc507818828"
UPSTREAM_ROOTS = []  # explicit existing TSM-native roots, never whole Drive
EXPLICIT_SOURCES = []  # optional exact {path, expected_sha256: known hash or None, lineage_ref}
PRODUCER_APPROVAL_PATH = ""  # external, personally signed; never created by this notebook
EXISTING_BINDING_PATH = ""  # optional, already produced development artifact
RELOCATED_DEVELOPMENT_ROOT = ""  # only when that artifact has moved
OUTPUT_PARENT = Path("/content/drive/MyDrive/00Quant_Research/data/development_snapshots")
REPORT_PARENT = Path("/content/drive/MyDrive/00Quant_Research/phase4a_preparation_reports")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import subprocess, sys, json, hashlib, re, importlib
from datetime import datetime, timezone

def checked(args):
    return subprocess.check_output(args, text=True).strip()

assert re.fullmatch('[0-9a-f]{40}', CORE_SHA) and re.fullmatch('[0-9a-f]{40}', AGENT_SHA)
subprocess.run([sys.executable, '-m', 'pip', 'install',
    'market-research-core @ git+https://github.com/hh4832/market-research-core.git@' + CORE_SHA], check=True)
AGENT_ROOT = Path('/content/strategy-research-agent-' + AGENT_SHA)
if not AGENT_ROOT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/hh4832/strategy-research-agent.git', str(AGENT_ROOT)], check=True)
    subprocess.run(['git', '-C', str(AGENT_ROOT), 'checkout', '--detach', AGENT_SHA], check=True)
assert checked(['git', '-C', str(AGENT_ROOT), 'rev-parse', 'HEAD']) == AGENT_SHA
assert not checked(['git', '-C', str(AGENT_ROOT), 'status', '--porcelain'])
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(AGENT_ROOT) + '[research]'], check=True)

# COLAB KERNEL IMPORT FIX:
# subprocess pip may install an editable-path hook that has not yet been
# activated in the running Colab interpreter. Use the pinned checkout directly.
agent_src = AGENT_ROOT / 'src'
assert (agent_src / 'strategy_agent' / '__init__.py').is_file(), (
    'Pinned Agent source package not found at ' + str(agent_src)
)
for directory in (AGENT_ROOT, agent_src):
    directory_str = str(directory)
    if directory_str not in sys.path:
        sys.path.insert(0, directory_str)
importlib.invalidate_caches()
from market_research_core.tsm_native import core_code_identity
from market_research_core.tsm_development import (
    inventory_isolated_sources, plan_conversion, produce_development_snapshot,
    preflight_development_snapshot, renew_development_receipt, file_sha,
)
from strategy_agent.research_only.development_snapshot import prepare_development_proposal
from strategy_agent import __file__ as agent_module_file
assert Path(agent_module_file).resolve().is_relative_to(AGENT_ROOT.resolve()), (
    'Agent resolved outside the pinned checkout: ' + str(agent_module_file)
)
print('Agent import: PASS', agent_module_file)
identity = core_code_identity()
assert identity['core_git_sha'] == CORE_SHA and identity['core_tree_dirty'] is not True, identity
REPORT_ROOT = REPORT_PARENT / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
REPORT_ROOT.mkdir(parents=True, exist_ok=False)

def save(name, value):
    with (REPORT_ROOT / name).open('x') as stream:
        json.dump(value, stream, indent=2, allow_nan=False); stream.write('\n')

def governed_fingerprint():
    paths = []
    for part in ('governance', 'registry', 'configs/core_cost_profiles', 'configs/research_only', 'configs/strategies'):
        base = AGENT_ROOT / part
        if base.exists(): paths.extend(p for p in base.rglob('*') if p.is_file())
    return {str(p.relative_to(AGENT_ROOT)): file_sha(p) for p in sorted(set(paths))}

def old_uses():
    path = AGENT_ROOT / 'registry/research_authorization_uses.jsonl'
    return sum(json.loads(line).get('authorization_id') == 'RA-IBKR_TSM-SMA520_SMOKE-0001'
               for line in path.read_text().splitlines() if line.strip())

BEFORE = governed_fingerprint()
USES_BEFORE = old_uses()
assert USES_BEFORE == 0, 'STOP: old authorization usage differs from accepted 0/1 baseline'
save('environment.json', {'core_sha': CORE_SHA, 'agent_sha': AGENT_SHA,
    'source_access': 'COLAB_DRIVE', 'governed_before': BEFORE, 'old_uses_before': USES_BEFORE})


In [ ]:
# Metadata-only. No full source checksum / OHLCV decoding in this cell.
bundle = None
plan = None
if EXISTING_BINDING_PATH:
    original_bundle = json.loads(Path(EXISTING_BINDING_PATH).read_text())
    if RELOCATED_DEVELOPMENT_ROOT:
        original_bundle['snapshot_root'] = RELOCATED_DEVELOPMENT_ROOT
    # Only pinned development-only bytes can be hashed; ALL footer boundaries checked first.
    bundle = renew_development_receipt(original_bundle)
    save('renewed_binding.json', bundle)  # outside immutable Snapshot
else:
    manifest = SOURCE_ROOT / 'final_gate_manifest.json'
    assert file_sha(manifest) == PARENT_MANIFEST_SHA, 'STOP: parent manifest mismatch'
    parent = json.loads(manifest.read_text())
    tsm = [r for r in parent['results'] if r.get('symbol') == 'TSM']
    assert len(tsm) == 1, 'STOP: ambiguous parent symbol record'
    # Source lineage hints only; these fields do not prove a chunk is safe.
    hints = {k: v for k, v in tsm[0].items() if 'source' in k.lower() or 'path' in k.lower()}
    print('Recorded upstream hints:', hints)
    inventory = inventory_isolated_sources(UPSTREAM_ROOTS,
        parent_manifest_path=str(manifest), parent_manifest_sha256=PARENT_MANIFEST_SHA)
    save('source_inventory.json', inventory)
    sources = EXPLICIT_SOURCES or inventory['safe_sources']
    if not sources:
        # Fallback PLAN ONLY; cannot execute without exact personal producer approval.
        sources = [{'path': str(SOURCE_ROOT / 'tsm_normalized.parquet'),
            'expected_sha256': tsm[0]['published_sha256'],
            'lineage_ref': str(manifest) + '#TSM.published_sha256 (claim until independently hashed)'}]
    plan = plan_conversion(sources, parent_manifest_path=manifest,
        expected_parent_manifest_sha256=PARENT_MANIFEST_SHA)
    save('producer_plan.json', plan)
    save('producer_approval.UNSIGNED.json', {'status': 'PROPOSED_UNSIGNED',
        'approved_by': '__RESEARCH_LEAD__', 'approval_ref': '__PERSONALLY_SET_BY_LEAD__',
        'plan_sha256': plan['plan_sha256'], 'allow_full_source_opaque_hash': False,
        'allow_holdout_decode_for_conversion_only': False, 'allow_strategy_run': False})
    print(json.dumps(plan, indent=2))
    if plan['requires_research_lead_approval']:
        print('PENDING_LEAD_APPROVAL: full-file hashing may read Holdout bytes; selected-group decoding may materialize Holdout OHLCV. No conversion performed yet.')


In [ ]:
# No approval is inferred from running this cell. Producer rederives footer risk facts.
if bundle is None:
    if plan['requires_research_lead_approval'] and not PRODUCER_APPROVAL_PATH:
        print('STOPPED SAFELY: review exact plan; obtain personal producer approval before conversion.')
    else:
        approval = json.loads(Path(PRODUCER_APPROVAL_PATH).read_text()) if PRODUCER_APPROVAL_PATH else None
        bundle = produce_development_snapshot(plan, OUTPUT_PARENT, producer_sha=CORE_SHA, approval=approval)
        save('produced_binding.json', bundle)

if bundle is not None:
    real_report = preflight_development_snapshot(bundle)
    save('real_snapshot_preflight.json', real_report)
    proposal = prepare_development_proposal(bundle, core_sha=CORE_SHA,
        output_directory=REPORT_ROOT / 'new_unsigned_research_proposal', root=AGENT_ROOT)
    assert proposal['slot_consumed'] is False and proposal['eligible_to_backtest'] is False
    print(json.dumps(proposal, indent=2))
else:
    real_report = {'status': 'NOT_RUN_PENDING_SAFE_SOURCE_OR_PERSONAL_CONVERSION_APPROVAL'}

AFTER = governed_fingerprint()
assert AFTER == BEFORE, 'STOP: governed files changed'
assert old_uses() == USES_BEFORE == 0
summary = {'core_sha': CORE_SHA, 'agent_sha': AGENT_SHA,
    'real_development_snapshot_created_or_revalidated': bundle is not None,
    'preflight': real_report, 'formal_gate': 'UNCHANGED', 'research_authorization': 'NOT_SIGNED',
    'batch_usage_before': USES_BEFORE, 'batch_usage_after': old_uses(),
    'strategy_run': False, 'eligible_to_backtest': False,
    'next_action': 'Personal approval of NEW snapshot authorization after data PASS; normal non-consuming Agent checks before separately requested research run'}
save('preparation_summary.json', summary)
print(json.dumps(summary, indent=2))
print('Reports:', REPORT_ROOT)


## Stop

Do not call any backtest or signing API here. If the report is pending, the only conversion permission needed is the **exact generated plan**, conditional on no usable isolated source. Its full-file hash includes original Holdout bytes and mixed-group decode can include Holdout OHLCV; output is development-only. After a real preflight PASS, the separate NEW authorization must be approved personally. Original SMOKE-0001 stays 0/1.

This notebook has no precomputed real-data outputs. CI validates code/synthetic fixtures; the Drive run supplies the actual evidence. Never relabel synthetic tests as real Snapshot acceptance.
